# Evaluate existing service performance

Read the operational records and service-intention mapping, check the selected jurisdictions, and estimate group and meta-frontier performance. The packaged case contains 200 DMUs. The DDF uses SciPy HiGHS and does not require a Gurobi licence. Complete this stage before fitting the Tobit models.

Select a Python kernel with the required packages; the saved ML model requires scikit-learn 1.5.1 and XGBoost 2.1.0. The reference installation uses the **ArcPy** kernel (Anaconda base). Run the cells in order. Shared settings are in `desktop_settings.json`; the default result folder is `../outputs/desktop_run_en`. The scripts check dependencies without installing packages.

In [1]:
from pathlib import Path
import sys
# Locate the toolkit from the working folder, its parents, or the configured fallback.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents,
    Path(r'E:\On-demand Transport Service\Analysis\7_Toolkit')]
    if (p/'desktop_scripts/desktop_helpers.py').is_file()), None)
if ROOT is None: raise FileNotFoundError('Cannot locate the toolkit folder')
sys.path.insert(0, str(ROOT/'desktop_scripts'))
from desktop_helpers import check_environment, run, run_r, OUT
import pandas as pd
versions = check_environment()

Existing Python: C:\Users\USER\anaconda3\python.exe
Package versions: {'numpy': '1.26.4', 'pandas': '2.3.0', 'scipy': '1.15.2', 'sklearn': '1.5.1', 'xgboost': '2.1.0', 'joblib': '1.4.2'}
Results: E:\On-demand Transport Service\Analysis\7_Toolkit\outputs\desktop_run_en


Inspect the selected jurisdictions and service types.

In [2]:
historical = pd.read_csv(ROOT/'data/evaluation/Model Data.csv')
print(historical.groupby('COUNTYNAME').size())
print(historical.ServiceType.value_counts())

COUNTYNAME
Hualien County     31
Kaohsiung City     68
New Taipei City    20
Tainan City        31
Taitung County     41
Yilan County        9
dtype: int64
ServiceType
Flexible Fixed-Route    91
Fixed-Route             80
Point-to-Point          29
Name: count, dtype: int64


Run validation and historical evaluation. The mathematical implementation is in drt_toolkit/evaluation.py.

In [3]:
run('validate', 'evaluate')
scores = pd.read_csv(OUT/'historical_scores.csv')
print(scores.head(12).to_string(index=False))


Stage: validate
Running validate ...
Completed: E:\On-demand Transport Service\Analysis\7_Toolkit\outputs\desktop_run_en

Stage: evaluate
Running evaluate ...
Completed: E:\On-demand Transport Service\Analysis\7_Toolkit\outputs\desktop_run_en
   DMU ClusterName  MetaBeta  GroupBeta  MetaEff  GroupEff      TGR  MaxResidual
ILA_01          FR  0.000000   0.000000 1.000000  1.000000 1.000000 1.088019e-14
ILA_02          FR  0.000000   0.000000 1.000000  1.000000 1.000000 6.661338e-16
ILA_03          FR  0.026889   0.000000 0.973111  1.000000 0.973111 6.439294e-15
ILA_04          FR  0.000000   0.000000 1.000000  1.000000 1.000000 6.661338e-16
ILA_05          FR  0.077114   0.000000 0.922886  1.000000 0.922886 0.000000e+00
ILA_06          FR  0.000000   0.000000 1.000000  1.000000 1.000000 5.995204e-15
ILA_07          FR  0.026917   0.000000 0.973083  1.000000 0.973083 5.684342e-14
ILA_08          FR  0.000000   0.000000 1.000000  1.000000 1.000000 4.574119e-14
ILA_09          FR  0.00000

Fit the Tobit models by running this cell, or open 02_Performance_Drivers.R in RStudio and click Source. Use one method for this run ID.

In [4]:
run_r('02_Performance_Drivers.R')
print((OUT/'tobit_summary.txt').read_text(encoding='utf-8',errors='replace'))

$FR

Call:
AER::tobit(formula = GroupEff ~ SI_Education + SI_Elderly + SI_Indigenous + 
    SI_Medicare + PTOverlap_per + Elderly_per + DayworkPopDensity + 
    Operator_Bus + Operator_Taxi + Operator_NGO, left = 0, right = 1, 
    data = z)

Observations:
         Total  Left-censored     Uncensored Right-censored 
            80              0              7             73 

Coefficients:
                   Estimate Std. Error z value Pr(>|z|)    
(Intercept)         0.93605    0.10930   8.564   <2e-16 ***
SI_Education        0.02914    0.04295   0.679    0.497    
SI_Elderly          0.19211  242.81743   0.001    0.999    
SI_Indigenous       0.03644    0.04860   0.750    0.453    
SI_Medicare        -0.03795    0.04699  -0.808    0.419    
PTOverlap_per      -0.03438    0.07383  -0.466    0.641    
Elderly_per         0.17774    0.35697   0.498    0.619    
DayworkPopDensity   0.09667    0.07007   1.379    0.168    
Operator_Bus        0.03868    0.06132   0.631    0.528    
Operat